# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² clinical oncology dataset using the `mlcroissant` library.

### Dataset Source
The dataset metadata and resources are described by a [Croissant schema](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) from SenScience.

In [ ]:
# Ensure `mlcroissant` is installed (restart runtime if needed)
!pip install -U mlcroissant

## 1. Data Loading
Load the Croissant metadata and initialize the dataset object.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the Croissant dataset object
dataset = mlc.Dataset(croissant_url)

# Show dataset metadata
meta = dataset.metadata
print(f"Dataset name: {meta.name}\nDescription: {meta.description}")

## 2. Data Overview
Review available record sets (tables) and their fields (columns), referencing by their `@id` fields.

In [ ]:
# List all record sets and their fields by `@id`
print("Record sets available in this dataset:")
recordsets = dataset.metadata.record_sets
if not recordsets:
    print("No record sets defined in the metadata. Attempting to parse from resources...")

# Fallback: try to infer record set IDs from dataset.records API (if metadata.record_sets empty)
try:
    record_set_ids = dataset.list_record_sets()
except AttributeError:
    # Older mlcroissant, iterate once to get IDs
    record_set_ids = set()
    try:
        _ = list(dataset.records())
    except Exception as e:
        print("Could not enumerate records; check dataset schema.")
        record_set_ids = list(record_set_ids)

if recordsets:
    for rs in recordsets:
        print(f"- {rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs}")
        # Fields may be present as 'field' or 'fields' attribute
        fields = rs.get('field', []) if isinstance(rs, dict) else []
        if not hasattr(fields, '__iter__') or isinstance(fields, str):
            fields = [fields]
        for f in fields:
            if isinstance(f, dict) and '@id' in f:
                print(f"    - Field: {f['@id']}")
            else:
                print(f"    - Field: {f}")
elif record_set_ids:
    for rset_id in record_set_ids:
        print(f"- {rset_id}")
        try:
            example = next(dataset.records(record_set=rset_id))
            for k in example.keys():
                print(f"    - Field: {k}")
        except Exception as e:
            print(f"    (Fields could not be enumerated for {rset_id})")
else:
    print("No record sets found.")

## 3. Data Extraction
Load data from the main record set into a DataFrame. All record set and field references use their `@id` values.

In [ ]:
# Identify the main record set by inspecting fields above; if known, set the appropriate @id
# For this dataset, we'll attempt to list all record sets and pick the first for demonstration

# Try to get all available record set @id's
try:
    record_set_ids = dataset.list_record_sets()
except AttributeError:
    # fallback: guess from known Croissant convention
    record_set_ids = []
    if dataset.metadata.record_sets:
        for rs in dataset.metadata.record_sets:
            if isinstance(rs, dict) and "@id" in rs:
                record_set_ids.append(rs["@id"])

if not record_set_ids:
    # Try known convention
    record_set_ids = [
        "https://api.app.sen.science/frontiers/7862866/second_primary_crc_records"
    ]

# Load each record set into a DataFrame
dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records from record set {record_set_id}")
    except Exception as e:
        print(f"Failed to load records from {record_set_id}\n{e}")

# Display columns and a preview for the main record set
main_record_set = record_set_ids[0]
if main_record_set in dataframes:
    print("Columns in main record set (by their @id):")
    print(list(dataframes[main_record_set].columns))
    dataframes[main_record_set].head()
else:
    print("No data loaded into DataFrame.")

## 4. Exploratory Data Analysis (EDA)
Apply typical exploratory data analysis steps: filter by a numeric field, normalize a column, and optionally group by another variable. All field references use their `@id` values.

In [ ]:
# Replace with actual numeric field's @id from previous output
# For example: 'age_at_second_crc' or a similar field
main_df = dataframes.get(main_record_set, pd.DataFrame())
print("Available columns in DataFrame:", list(main_df.columns))

# Try to pick a numeric field
sample_numeric_field = None
numeric_candidates = [
    c for c in main_df.columns if any([s in c.lower() for s in ["age", "interval", "years", "numeric", "count"]])
]
if numeric_candidates:
    sample_numeric_field = numeric_candidates[0]
else:
    # Fall back to first column (for demonstration)
    sample_numeric_field = list(main_df.columns)[0] if list(main_df.columns) else None
print(f"Selected numeric field for EDA: {sample_numeric_field}")

if sample_numeric_field:
    # Remove outliers/invalids
    filtered_df = main_df[pd.to_numeric(main_df[sample_numeric_field], errors='coerce') > 10]
    print(f"Filtered records with {sample_numeric_field} > 10:")
    print(filtered_df[[sample_numeric_field]].head())

    # Normalize
    vals = pd.to_numeric(filtered_df[sample_numeric_field], errors='coerce')
    filtered_df[f"{sample_numeric_field}_normalized"] = (vals - vals.mean()) / vals.std()
    print(f"Normalized {sample_numeric_field}:")
    print(filtered_df[[sample_numeric_field, f"{sample_numeric_field}_normalized"]].head())

    # Try grouping by a categorical field
    group_keys = [col for col in main_df.columns if col != sample_numeric_field]
    group_field = None
    for k in group_keys:
        if any(x in k.lower() for x in ["sex", "msi", "histology", "group", "site", "anatomical"]):
            group_field = k
            break
    print(f"Grouping field: {group_field}")
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[sample_numeric_field].mean()
        print(f"Mean {sample_numeric_field} grouped by {group_field}:")
        print(grouped_df)
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize a distribution or relationship using matplotlib.

In [ ]:
import matplotlib.pyplot as plt

if sample_numeric_field and not main_df.empty:
    fig, ax = plt.subplots(figsize=(7,4))
    pd.to_numeric(main_df[sample_numeric_field], errors='coerce').hist(bins=15, ax=ax, color='dodgerblue')
    ax.set_title(f"Distribution of {sample_numeric_field}")
    ax.set_xlabel(sample_numeric_field)
    ax.set_ylabel("Count")
    plt.show()
    
    # If group_field selected, make boxplot
    if group_field:
        plt.figure(figsize=(8,4))
        main_df.boxplot(column=sample_numeric_field, by=group_field)
        plt.title(f"{sample_numeric_field} by {group_field}")
        plt.suptitle("")
        plt.ylabel(sample_numeric_field)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to:

- Load a FAIR² clinical dataset described in Croissant format via its schema URL.
- Explore available record sets and fields referencing their `@id`.
- Extract and display data in Pandas DataFrames using mlcroissant.
- Perform initial EDA: filtering, normalization, and grouping by category fields.
- Visualize numeric field distributions and relationships.

Using Croissant's `@id` referencing makes the notebook robust and aligned to FAIR data principles and interdisciplinary reuse.

**You can now extend the analysis to statistical comparisons, survival analysis, or model-building as needed.**